In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from backtest.data import DataPanel, FieldSpec

# --- fetch prices + volume ---
raw = yf.download(['^GSPC', '^VIX'], start='1990-01-01', auto_adjust=False, progress=False)['Close']
raw.columns = ['SPX', 'VIX']
raw = raw.dropna()

vol_raw = yf.download('^GSPC', start='1990-01-01', auto_adjust=False, progress=False)['Volume'].squeeze()
vol_raw.name = 'SPX'

# align on common dates
idx = raw.index.intersection(vol_raw.index)
raw     = raw.loc[idx]
vol_raw = vol_raw.loc[idx]

# --- prices: SPX (price index) + synthetic CASH at 1.0 (zero interest) ---
prices_df = pd.DataFrame({
    'SPX':  raw['SPX'],
    'CASH': 1.0,
}, index=raw.index)

# --- volume: CASH gets NaN (addressed in Cell 3 — costs config) ---
volume_df = pd.DataFrame({
    'SPX':  vol_raw,
    'CASH': float('nan'),
}, index=raw.index)

# --- VIX feature: broadcast to match prices columns (DataPanel requires identical columns) ---
vix_df = pd.DataFrame({
    'SPX':  raw['VIX'],
    'CASH': raw['VIX'],
}, index=raw.index)

# --- panel ---
panel = DataPanel(
    prices_df,
    volume=volume_df,
    features={'VIX': vix_df},
    check_outliers=True,
)

print(f"Bars:   {len(panel.dates)}")
print(f"Assets: {list(panel.assets_all)}")
print(f"Range:  {panel.dates[0].date()} to {panel.dates[-1].date()}")
print(f"NaN fraction — prices: {prices_df.isna().mean().mean():.3%}")
print(f"NaN fraction — volume SPX: {volume_df['SPX'].isna().mean():.3%} (CASH intentionally NaN)")
prices_df.tail(3)

Bars:   9173
Assets: ['SPX', 'CASH']
Range:  1990-01-02 to 2026-06-04
NaN fraction — prices: 0.000%
NaN fraction — volume SPX: 0.000% (CASH intentionally NaN)


,SPX,CASH
Date,,
2026-06-02,7609.779785,1.0
2026-06-03,7553.680176,1.0
2026-06-04,7584.310059,1.0


In [2]:
import sys
import numpy as np
from pathlib import Path

_nb_dir = str(Path.cwd())
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

from _mood_helpers import _mood_d_max, calibrate_mood_threshold

MOOD_THRESHOLD = calibrate_mood_threshold()
print(f"\nMOOD_THRESHOLD in scope: {MOOD_THRESHOLD:.6f}")
# MOOD_THRESHOLD = 4.256531  (ARL=10000, n_sim=500, max_n=30000, seed=42)

JIT warm-up ... done

Calibrating  ARL=10,000  n_sim=500  max_n=30,000
iter        h     est_ARL    time
   1   4.7500       24123  1879.7s
   2   4.1250        6254  242.8s
   3   4.4375       15349  1061.9s
   4   4.2812       10338  603.0s
   5   4.2031        8263  373.5s
   6   4.2422        9066  368.8s
   7   4.2617       10033  444.0s
   8   4.2520        9637  502.1s
   9   4.2568        8640  428.7s
  10   4.2593        9714  456.8s
  11   4.2605        9885  390.9s
  12   4.2611       10079  461.9s

MOOD_THRESHOLD = 4.260803

MOOD_THRESHOLD in scope: 4.260803


In [4]:
import numpy as np
import pandas as pd
from backtest.strategy import Strategy
from backtest.risk.config import RiskConfig
from _mood_helpers import _mood_d_max_tau

class VIXChangePoint(Strategy):
    """VIX Mood CPD — rotate SPX/CASH based on EWMA volatility regime."""
    rebalance_frequency = "daily"
    risk = RiskConfig(max_position=None, max_gross=1.0, max_net=1.0)

    def __init__(
        self,
        ewma_lambda:   float = 0.95,
        vol_threshold: float = 0.20,
        warmup:        int   = 21,
        threshold:     float = MOOD_THRESHOLD,
    ):
        self.ewma_lambda   = ewma_lambda
        self.vol_threshold = vol_threshold
        self.warmup        = warmup
        self.threshold     = threshold
        self._reset_state()

    def __repr__(self):
        return (
            f"VIXChangePoint(ewma_lambda={self.ewma_lambda}, "
            f"vol_threshold={self.vol_threshold}, "
            f"warmup={self.warmup}, "
            f"threshold={self.threshold:.6f})"
        )

    def required_data(self):
        return {"prices": None, "VIX": None}

    # ── internal ──────────────────────────────────────────────────────────────

    def _reset_state(self):
        self._vix_buf  = []   # VIX log-rets since last CP (for CPD)
        self._spx_buf  = []   # SPX log-rets since last CP (for EWMA re-est.)
        self._ewma_var = None
        self._prev_w   = None # yesterday's proposed weight → 1-day exec lag

    def _ewma_from_scratch(self, rets):
        """Compute EWMA variance on a list, starting fresh."""
        lam = self.ewma_lambda
        v = rets[0] ** 2
        for r in rets[1:]:
            v = lam * v + (1 - lam) * r ** 2
        return v

    # ── engine interface ──────────────────────────────────────────────────────

    def generate_weights(self, data, t):
        vix_s   = data.feature("VIX")["SPX"]
        price_s = data.prices["SPX"]
        if len(vix_s) < 2 or len(price_s) < 2:
            return pd.Series({"SPX": 0.0, "CASH": 1.0})

        vix_ret = float(np.log(vix_s.iloc[-1]   / vix_s.iloc[-2]))
        spx_ret = float(np.log(price_s.iloc[-1] / price_s.iloc[-2]))

        self._vix_buf.append(vix_ret)
        self._spx_buf.append(spx_ret)

        # incremental EWMA (SPX returns)
        lam = self.ewma_lambda
        if self._ewma_var is None:
            self._ewma_var = spx_ret ** 2
        else:
            self._ewma_var = lam * self._ewma_var + (1 - lam) * spx_ret ** 2

        # CPD check — only once warmup observations have accumulated
        n = len(self._vix_buf)
        if n > self.warmup:
            buf_arr = np.array(self._vix_buf, dtype=np.float64)
            dmax, tau_f = _mood_d_max_tau(buf_arr, n)
            if dmax > self.threshold:
                tau = int(tau_f)
                # Re-estimate EWMA from the post-change-point SPX returns
                post = self._spx_buf[tau:]
                if post:
                    self._ewma_var = self._ewma_from_scratch(post)
                # Fresh CPD buffer for the new regime
                self._vix_buf = []
                self._spx_buf = []

        # Proposed weight for tomorrow (1-day execution lag)
        ann_vol = np.sqrt(max(self._ewma_var, 0.0) * 252)
        if ann_vol < self.vol_threshold:
            proposed = pd.Series({"SPX": 1.0, "CASH": 0.0})
        else:
            proposed = pd.Series({"SPX": 0.0, "CASH": 1.0})

        # Return previous bar's weight; store today's for next bar
        weight       = self._prev_w if self._prev_w is not None else pd.Series({"SPX": 0.0, "CASH": 1.0})
        self._prev_w = proposed
        return weight


strat = VIXChangePoint()
print(strat)
print(f"required_data : {strat.required_data()}")
print(f"Rebalances/yr : {len(strat.rebalance_dates(panel.dates)) / (len(panel.dates) / 252):.1f}")

VIXChangePoint(ewma_lambda=0.95, vol_threshold=0.2, warmup=21, threshold=4.260803)
required_data : {'prices': None, 'VIX': None}
Rebalances/yr : 252.0


In [5]:
from backtest.costs import (
    Commission, Spread, MarketImpact,
    CompositeCostModel, LiquidityCap,
)

costs = CompositeCostModel([
    Commission(bps=1),                                  # 1 bps per traded dollar (both legs)
    Spread(half_bps={'SPX': 1, 'CASH': 0}),            # 1 bps half-spread on SPX; CASH is synthetic
    MarketImpact(k=10, kind="sqrt", adv_lookback=20),  # sqrt impact; CASH NaN vol → 0 impact
])

liq = LiquidityCap(cap_pct=0.10, adv_lookback=20)     # 10 % of 20-day median ADV

print("Cost components :", [type(m).__name__ for m in costs.models])
print(f"LiquidityCap    : cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")
print(f"Panel has volume: {panel.has_field('volume')}")

Cost components : ['Commission', 'Spread', 'MarketImpact']
LiquidityCap    : cap_pct=0.1, adv_lookback=20
Panel has volume: True


In [6]:
from backtest.engine import Engine

engine = Engine(
    panel,
    VIXChangePoint(),           # fresh instance — no state from Cell 3 validation
    costs=costs,
    liquidity=liq,
    initial_capital=1_000_000,
)

result = engine.run()           # full history; strategy handles its own 21-bar warmup

print(f"Bars          : {len(result.equity_curve)}")
print(f"Rebalances    : {result.metadata['n_rebalances']}")
print(f"Date range    : {result.metadata['start'].date()} → {result.metadata['end'].date()}")
print(f"Initial       : ${result.metadata['initial_capital']:>14,.0f}")
print(f"Final equity  : ${result.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return  : {(result.equity_curve.iloc[-1] / result.metadata['initial_capital'] - 1):+.2%}")
print(f"Total costs   : ${result.costs.sum():>14,.0f}")
print(f"Max pos weight: {result.weights.abs().max().max():.2%}")
print(f"Max gross exp : {result.weights.abs().sum(axis=1).max():.2%}")
print(f"Unfilled (any): {(result.unfilled.abs() > 0).any().any()}")

Bars          : 9173
Rebalances    : 9173
Date range    : 1990-01-02 → 2026-06-04
Initial       : $     1,000,000
Final equity  : $     7,566,394
Total return  : +656.64%
Total costs   : $       138,058
Max pos weight: 100.03%
Max gross exp : 100.03%
Unfilled (any): False


In [7]:
from backtest.splitters import CombinatorialPurgedCV

splitter = CombinatorialPurgedCV(
    n_splits=6,
    n_test_groups=2,
    purge_bars=5,
    embargo_pct=0.01,
)

n = len(panel.dates)
embargo_bars = int(n * splitter.embargo_pct)
print(f"Timeline:           {n} bars  ({panel.dates[0].date()} → {panel.dates[-1].date()})")
print(f"Groups:             {splitter.n_groups} contiguous groups of ~{n // splitter.n_groups} bars each")
print(f"Splits (trainings): {splitter.n_splits()}")
print(f"Paths (curves):     {splitter.n_paths()}")
print(f"Purge:              {splitter.purge_bars} bars on each side of every test run")
print(f"Embargo:            {embargo_bars} bars after each test run  (trailing buffer = {splitter.purge_bars + embargo_bars})")

first_train, first_test = next(iter(splitter.split(panel.dates)))
print(f"\nFirst split:  train={len(first_train)}, test={len(first_test)}, disjoint={len(first_train.intersection(first_test)) == 0}")

Timeline:           9173 bars  (1990-01-02 → 2026-06-04)
Groups:             6 contiguous groups of ~1528 bars each
Splits (trainings): 15
Paths (curves):     5
Purge:              5 bars on each side of every test run
Embargo:            91 bars after each test run  (trailing buffer = 96)

First split:  train=6020, test=3057, disjoint=True


In [8]:
from backtest.engine import MultiPathEngine

mp = MultiPathEngine(
    panel,
    lambda: VIXChangePoint(),    # factory — fresh state per split
    splitter,
    costs=costs,
    liquidity=liq,
    initial_capital=1_000_000,
)
res = mp.run(n_jobs=-1)

print(f"Splits run:           {res.n_splits}")
print(f"Paths assembled:      {res.n_paths}")
print(f"path_returns shape:   {res.path_returns.shape}")
print(f"path_equity shape:    {res.path_equity.shape}")
print(f"Date range:           {res.path_returns.index[0].date()} → {res.path_returns.index[-1].date()}")
print(f"Per-path NaN count:   min={res.path_returns.isna().sum().min()}, max={res.path_returns.isna().sum().max()}")
print()
print("Final equity per path:")
print(res.path_equity.iloc[-1].rename("final_equity").map(lambda v: f"${v:,.0f}"))

Splits run:           15
Paths assembled:      5
path_returns shape:   (9173, 5)
path_equity shape:    (9173, 5)
Date range:           1990-01-02 → 2026-06-04
Per-path NaN count:   min=5, max=5

Final equity per path:
path_0    $7,642,727
path_1    $7,673,312
path_2    $7,384,377
path_3    $7,674,875
path_4    $7,193,197
Name: final_equity, dtype: object


In [9]:
from backtest.metrics import compute_metrics

rep = compute_metrics(
    result.returns,
    result.equity_curve,
    costs=result.costs,
    trades=result.trades,
    ann_factor=252,
    ci_level=0.95,
)
print(rep)
print()
print(f"Sharpe:    {rep.sharpe:.3f}  ± {rep.sharpe_std:.3f}  "
      f"(95% CI: [{rep.sharpe_ci_low:.3f}, {rep.sharpe_ci_high:.3f}])")
print(f"PSR(0):    {rep.psr:.3f}  {'significant' if rep.psr > 0.95 else 'NOT significant'} at 95%")
print(f"MinTRL(0): {rep.min_trl:.0f} bars  (sample: {rep.n_obs} bars  "
      f"→ {'powered' if rep.min_trl < rep.n_obs else 'underpowered'})")

print("\nYearly breakdown:")
print(rep.yearly_table())

print("\n--- CPCV aggregate ---")
agg = res.aggregate_metrics()
print(f"Sharpe  mean={agg['sharpe_mean']:.3f}  std={agg['sharpe_std']:.3f}  "
      f"min={agg['sharpe_min']:.3f}  max={agg['sharpe_max']:.3f}")
print(f"Calmar  mean={agg['calmar_mean']:.3f}  std={agg['calmar_std']:.3f}")
print(f"Max DD  mean={agg['max_drawdown_mean']:.3f}  std={agg['max_drawdown_std']:.3f}")

MetricsReport:
  Total return         +656.72%
  Annualized return    +5.71%
  Annualized vol       11.72%
  Hit rate             42.23%
  Total PnL ($)        +6,566,494.46
  Total costs ($)      138,057.80
  Gross PnL ($)        +6,704,552.25
  Turnover             302.54x
  Margin               0.7155%
  Sharpe (ann.)        0.533
  Sharpe SD (ann.)     0.168
  Sharpe 95% CI        [0.205, 0.862]
  Sortino              0.454
  Calmar               0.135
  Modified Sharpe      5.094
  PSR (vs 0)           0.999
  MinTRL (vs 0)        2453 bars
  VaR 5%               -0.0123
  cVaR 5%              -0.0188
  Max drawdown         46.35%
  Time underwater      94.08%
  Longest underwater   3704 bars
  N obs                9172
  Median cal-year ret  +8.99%
  Median arith ann     +9.23%

Sharpe:    0.533  ± 0.168  (95% CI: [0.205, 0.862])
PSR(0):    0.999  significant at 95%
MinTRL(0): 2453 bars  (sample: 9172 bars  → powered)

Yearly breakdown:
        n  total_return  arith_ann    sharp

In [11]:
from backtest.metrics import compute_metrics

# --- last calendar month (relative to final backtest bar) ---
_end   = result.equity_curve.index[-1]
_start = (_end - pd.DateOffset(months=1)) + pd.Timedelta(days=1)
_mask  = result.equity_curve.index >= _start

_last_rets   = result.returns.loc[_mask]
_last_equity = result.equity_curve.loc[_mask]
_last_costs  = result.costs.loc[_mask]
_last_trades = result.trades.loc[_mask]

print(f"Last-month window : {_start.date()} → {_end.date()}  ({_mask.sum()} bars)")
print(f"Start equity      : ${_last_equity.iloc[0]:>14,.0f}")
print(f"End equity        : ${_last_equity.iloc[-1]:>14,.0f}")
print(f"Period return     : {(_last_equity.iloc[-1] / _last_equity.iloc[0] - 1):+.2%}")
print(f"Period costs      : ${_last_costs.sum():>14,.0f}")
print(f"Avg SPX weight    : {result.weights.loc[_mask, 'SPX'].mean():.1%}")
print(f"Avg gross exp     : {result.weights.loc[_mask].abs().sum(axis=1).mean():.1%}")
print()

rep_last = compute_metrics(
    _last_rets,
    _last_equity,
    costs=_last_costs,
    trades=_last_trades,
    ann_factor=252,
    ci_level=0.95,
)
print(rep_last)
print()
print(f"Sharpe (ann.) : {rep_last.sharpe:.3f}  ± {rep_last.sharpe_std:.3f}  "
      f"(95% CI: [{rep_last.sharpe_ci_low:.3f}, {rep_last.sharpe_ci_high:.3f}])")
print(f"Max drawdown  : {rep_last.max_drawdown:.2%}")
print(f"Hit rate      : {rep_last.hit_rate:.2%}")
print(f"N obs         : {rep_last.n_obs}")

Last-month window : 2026-05-05 → 2026-06-04  (22 bars)
Start equity      : $     7,242,073
End equity        : $     7,566,394
Period return     : +4.48%
Period costs      : $             0
Avg SPX weight    : 100.0%
Avg gross exp     : 100.0%

MetricsReport:
  Total return         +4.48%
  Annualized return    +70.47%
  Annualized vol       9.95%
  Hit rate             72.73%
  Total PnL ($)        +324,321.92
  Total costs ($)      0.00
  Gross PnL ($)        +324,321.92
  Turnover             0.00x
  Sharpe (ann.)        6.032
  Sharpe SD (ann.)     3.887
  Sharpe 95% CI        [-1.588, 13.651]
  Sortino              5.640
  Calmar               30.480
  Modified Sharpe      81.756
  PSR (vs 0)           0.940
  MinTRL (vs 0)        25 bars
  VaR 5%               -0.0073
  cVaR 5%              -0.0099
  Max drawdown         1.97%
  Time underwater      45.45%
  Longest underwater   6 bars
  N obs                22
  Median cal-year ret  +5.33%
  Median arith ann     +59.99%

Sharpe 

In [10]:
from backtest.selection import dsr, effective_k, expected_max_sr
from backtest.metrics.core import sharpe_var_term
import numpy as np

K = 1  # first and only trial of VIXChangePoint this session

deflated = dsr(result.returns, K=K)

r = result.returns.dropna().to_numpy()
T = len(r)
sr_pb    = r.mean() / r.std(ddof=1)
var_term = sharpe_var_term(r, sr_pb)
luck_max_ann = expected_max_sr(K, var_term / T) * np.sqrt(252)

print(f"Trials (K):             {K}")
print(f"Realized Sharpe (ann.): {sr_pb * np.sqrt(252):.3f}")
print(f"Luck-max threshold:     {luck_max_ann:.3f}")
print(f"DSR:                    {deflated:.3f}   "
      f"{'significant' if deflated > 0.95 else 'NOT significant'} at 95%")

deflated_paths = [dsr(res.path_returns[c].dropna(), K=K) for c in res.path_returns.columns]
print(f"\nCPCV per-path DSR (K={K}):  "
      f"median={np.median(deflated_paths):.3f}  "
      f"min={np.min(deflated_paths):.3f}  max={np.max(deflated_paths):.3f}")

K_eff = effective_k(res.path_returns.dropna(how='all'), threshold=0.5)
print(f"Effective-K from CPCV paths: {K_eff}  "
      f"(paths are not independent trials — K stays {K})")

Trials (K):             1
Realized Sharpe (ann.): 0.533
Luck-max threshold:     0.000
DSR:                    0.999   significant at 95%

CPCV per-path DSR (K=1):  median=0.999  min=0.999  max=0.999
Effective-K from CPCV paths: 1  (paths are not independent trials — K stays 1)
